# Figure6_rank8_program_distinctness_interpretability

In [1]:

from pathlib import Path
import os, re, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import hypergeom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset, random_split
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print('Torch unavailable:', e)

try:
    from tensorly.decomposition import parafac
    from tensorly.cp_tensor import cp_to_tensor
    HAS_TENSORLY = True
except Exception as e:
    HAS_TENSORLY = False
    print('Tensorly unavailable:', e)

project_dir = Path('/Users/sidaye/Documents/python/ST_MultiCAST')
input_dir = project_dir / 'Input'
base_output_dir = project_dir / 'Output'
model_comparison_dir = base_output_dir / 'model comparison'
ai_output_dir = base_output_dir / 'AI_spatiotemporal_models_python'
Spacepoints = ['st','SI1','SI2','SI3','SI4','SI5','SI6','SI7','SI8','SI9','ce','co']
Full_Timepoints = ['1h','3h','6h','12h','24h']
feature_order = [f'{t}_{s}' for t in Full_Timepoints for s in Spacepoints]

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'DejaVu Sans'

def save_pdf(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches='tight', transparent=True)
    plt.close(fig)

def build_data():
    df = pd.read_csv(input_dir / 'Spatial_temporal_MultiSCAST_FC_final_capping.csv')
    df['Gene'] = df['Gene'].astype(str)
    df['Time'] = df['Time'].astype(str)
    df['Space'] = df['Space'].astype(str)
    df = df[df['Time'].isin(Full_Timepoints) & df['Space'].isin(Spacepoints)].copy()
    df['Feature'] = df['Time'] + '_' + df['Space']
    df = df.groupby(['Gene','Time','Space','Feature'], as_index=False).agg(logFC=('logFC','mean'))
    wide = df.pivot_table(index='Gene', columns='Feature', values='logFC', aggfunc='mean')
    wide = wide[[f for f in feature_order if f in wide.columns]].dropna(axis=0, how='any')
    X_raw_df = wide.copy()
    X_raw = X_raw_df.values.astype(float)
    row_mean = X_raw.mean(axis=1, keepdims=True)
    row_std = X_raw.std(axis=1, keepdims=True)
    row_std[row_std == 0] = 1.0
    X_scaled = np.nan_to_num((X_raw - row_mean) / row_std)
    X_scaled_df = pd.DataFrame(X_scaled, index=X_raw_df.index.astype(str), columns=X_raw_df.columns)
    return X_raw_df, X_scaled_df, row_mean, row_std

def vector_to_landscape(vector, columns=None):
    if columns is None:
        columns = feature_order
    s = pd.Series(np.asarray(vector, dtype=float), index=columns)
    return s.reindex(feature_order).values.reshape(len(Full_Timepoints), len(Spacepoints))

def load_category_table():
    cat = pd.read_excel(input_dir / 'putative_driver_gene_categories_12class.xlsx', sheet_name=0)
    cat['locus_ID'] = cat['locus_ID'].astype(str)
    col = 'Putative_driver_category'
    cat_map = cat.set_index('locus_ID')[col].dropna().to_dict()
    cats = sorted(pd.Series(cat_map).dropna().unique().tolist())
    palette = plt.cm.tab20(np.linspace(0, 1, max(20, len(cats))))
    color_map = {c: palette[i] for i, c in enumerate(cats)}
    default = '#2f6db3'
    return cat_map, color_map, default

def load_annotation():
    ann = pd.read_csv(input_dir / 'new_annotations_with_uniprot_names.csv')
    ann['locus_ID'] = ann['locus_ID'].astype(str)
    display_cols = ['gene_name','uniprot_gene_name','gene_name_old','KEGG_VC_number']
    def display(row):
        for c in display_cols:
            v = row.get(c, np.nan)
            if pd.notna(v) and str(v).strip() and str(v).lower() != 'nan':
                return str(v)
        return str(row['locus_ID'])
    ann['Gene_display'] = ann.apply(display, axis=1)
    text_cols = [c for c in ann.columns if c != 'locus_ID']
    ann['Annotation_text'] = ann[text_cols].astype(str).replace('nan','', regex=False).agg(' | '.join, axis=1)
    return ann

def phase_for_time(t):
    return {'1h':'Early','3h':'Early','6h':'Middle','12h':'Middle','24h':'Late'}.get(t, '')

def niche_for_space(s):
    if s == 'st': return 'stomach'
    if str(s).startswith('SI'): return 'small_intestine'
    if s == 'ce': return 'cecum'
    if s == 'co': return 'colon'
    return s

outdir = base_output_dir / 'Program_distinctness6'
outdir.mkdir(parents=True, exist_ok=True)
corr=pd.read_csv(model_comparison_dir/'PCA_CP_VAE_CPVAE_rank2_to_rank10_program_distinctness_correlations_long.csv')
rank=8
models=['PCA','CP','VAE','weighted CPVAE']
fig,axes=plt.subplots(1,4,figsize=(13.4,3.5))
summary=[]
for ax,model in zip(axes,models):
    sub=corr[(corr['Rank'].eq(rank)) & (corr['Model'].eq(model))]
    mat=sub.pivot(index='Program_1',columns='Program_2',values='Correlation')
    labels=sub['Program_1'].drop_duplicates().tolist(); mat=mat.reindex(index=labels,columns=labels)
    im=ax.imshow(mat.values,cmap='coolwarm',vmin=-1,vmax=1)
    ax.set_title(model,fontsize=9,pad=7); ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels,rotation=45,ha='right',fontsize=6); ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels,fontsize=6)
    vals=mat.values; off=vals[~np.eye(vals.shape[0],dtype=bool)]; summary.append({'Model':model,'Rank':rank,'Mean_abs_offdiag_correlation':np.mean(np.abs(off)),'Distinctness_score':1-np.mean(np.abs(off))})
fig.suptitle('Rank8 program-effect correlation matrices',fontsize=12,y=0.98)
fig.subplots_adjust(top=0.78,bottom=0.22,left=0.05,right=0.88,wspace=0.50)
cax = fig.add_axes([0.905, 0.27, 0.014, 0.46])
fig.colorbar(im, cax=cax, label='Program correlation')
save_pdf(fig,outdir/'Figure6_rank8_program_correlation_matrices.pdf')
summ=pd.DataFrame(summary); summ.to_csv(outdir/'Figure6_rank8_program_distinctness_scores.csv',index=False)
fig,ax=plt.subplots(figsize=(5.8,3.4))
ax.bar(summ['Model'],summ['Distinctness_score'],color=['#4C72B0','#DD8452','#8172B2','#2A9D8F'])
ax.set_ylabel('Distinctness score (1 - mean |off-diagonal r|)'); ax.set_ylim(0,1); ax.set_title('Rank8 program distinctness',fontsize=11,pad=8); ax.tick_params(axis='x',rotation=20)
fig.subplots_adjust(top=0.86,bottom=0.24,left=0.16,right=0.96); save_pdf(fig,outdir/'Figure6_rank8_program_distinctness_score_barplot.pdf')
print(outdir)


/Users/sidaye/Documents/python/ST_MultiCAST/Output/Program_distinctness6
